In [0]:
from pyspark.sql.functions import *

# ==========================================
# S3 PATHS
# ==========================================

incoming_path = "s3://workflowautomationgiri/incoming/"
processed_path = "s3://workflowautomationgiri/processed/"

# ==========================================
# GET TXT FILES
# ==========================================

all_files = dbutils.fs.ls(incoming_path)

txt_files = [
    file.path
    for file in all_files
    if file.path.endswith(".txt")
]

# ==========================================
# READ PROCESSED LOG
# ==========================================

processed_df = spark.table(
    "workauto.gold.processed_files_log"
).filter(
    col("file_type") == "txt"
)

processed_files = [
    row["file_name"]
    for row in processed_df.collect()
]

# ==========================================
# FILTER NEW FILES
# ==========================================

new_files = []

for file in txt_files:

    file_name = file.split("/")[-1]

    if file_name not in processed_files:
        new_files.append(file)

# ==========================================
# PROCESS FILES
# ==========================================

if len(new_files) > 0:

    txt_df = spark.read.text(new_files)

    txt_df = txt_df.withColumnRenamed(
        "value",
        "raw_text"
    )

    txt_df = (
        txt_df
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "ingestion_time",
            current_timestamp()
        )
    )

    (
        txt_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.bronze.txt_raw")
    )

    # UPDATE LOG
    processed_log_df = spark.createDataFrame(
        [
            (file.split("/")[-1], "txt")
            for file in new_files
        ],
        ["file_name", "file_type"]
    ).withColumn(
        "processed_time",
        current_timestamp()
    )

    (
        processed_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.gold.processed_files_log")
    )

    # MOVE FILES
    for file in new_files:

        file_name = file.split("/")[-1]

        dbutils.fs.mv(
            file,
            processed_path + file_name
        )

    print("TXT Files Processed Successfully")

else:

    print("No New TXT Files Found")